# Voces y documentos peruanos · Evaluación

**Fecha: 25 de octubre de 2026, Perú.** Fuentes: corpus real de la Parte 1, Wikimedia Commons/BNP y OpenSLR 73, con atribuciones en los manifiestos. Modelos: PaddleOCR 3.7.0 vs. EasyOCR 1.7.2 (CER), faster-whisper `small` CPU vs. Qwen3-ASR 0.6B GPU (WER y latencia). LLM: Ollama local `qwen2.5:3b`.

La muestra de 10 documentos se estratifica 3 impresos/3 fotos/4 manuscritos y maximiza fuentes distintas. La muestra de 10 audios incluye las dos entrevistas naturales y ocho clips OpenSLR. Semilla: 20261025. Se guardan hashes de los archivos para comprobar que todos los modelos usan los mismos casos. Los parámetros se fijaron antes de consultar métricas.

**No se consideran referencias manuales ni auditoría humana las respuestas generadas por IA. Cuando falten, este notebook lo indica y no produce métricas finales.**

In [1]:
import json, pandas as pd
from pathlib import Path
from IPython.display import display
import plotly.express as px
from evaluacion import *
configuracion = json.loads((BASE/'modelos/configuracion.json').read_text(encoding='utf-8'))
display(configuracion)
muestra = preparar_muestra()
display(muestra[['id','modalidad','tipo','sha256_archivo']])
print('Referencias manuales completas:', int(muestra.texto_referencia.str.strip().ne('').sum()), '/20')


{'fecha_fijacion': '2026-10-25',
 'metrica_previa_consultada': False,
 'ocr_base': {'paquete': 'paddleocr',
  'version': '3.7.0',
  'detector': 'PP-OCRv6_medium_det',
  'reconocedor': 'PP-OCRv6_medium_rec',
  'lang': 'es',
  'device': 'cpu',
  'enable_mkldnn': False,
  'orientacion': False},
 'ocr_alterno': {'modelo': 'EasyOCR',
  'version': '1.7.2',
  'idiomas': ['es', 'en'],
  'gpu': False},
 'asr_base': {'modelo': 'faster-whisper small',
  'version': '1.2.1',
  'device': 'cpu',
  'compute_type': 'int8',
  'beam_size': 5,
  'vad_filter': True},
 'asr_alterno': {'modelo': 'Qwen/Qwen3-ASR-0.6B-hf',
  'transformers': '5.16.0',
  'device': 'cuda',
  'dtype': 'bfloat16',
  'VAD': 'Silero, max_tramo=12 s'},
 'nota': 'Comparación con muestra congelada antes de referencias; diferencias CPU/GPU deben declararse.'}

Muestra congelada conservada.


,id,modalidad,tipo,sha256_archivo
0,doc_005,documento,impreso,080beae4dff282f9f3183656aa2662031484c419ae8089...
1,doc_001,documento,impreso,b7da93e4c7db55869ab695c8d2ddf93cea6e0765d8078f...
2,doc_004,documento,impreso,f163caad346ce9e0499842dd2d5308f8c135ae92772c63...
3,doc_013,documento,foto,1caed506aac2d3e0f5d05438078bf8fed206accd589231...
4,doc_009,documento,foto,59c508a6611ab58c0388df63eb3f5b629e238ef6692738...
5,doc_010,documento,foto,f0dce587b5f339824e72664cb9385764fc5dc2dc68421a...
6,doc_018,documento,manuscrito,8ee2889dc33db1834aea7d7b805ba7f228a31d6fc54181...
7,doc_021,documento,manuscrito,550a471c7dfa382a9222587b346a8f828303cd725eb009...
8,doc_022,documento,manuscrito,4f5f5726eb98f0d31cb115c51ffeb0a14d2a0b5e8e656d...
9,doc_023,documento,manuscrito,a5ffab1acdf88b36ef7e26d7ae140f17265e5fad02f639...


Referencias manuales completas: 20 /20


## Inferencia separada de las referencias

Las predicciones se generan sin mirar referencias manuales ni métricas y se conservan como borradores privados. Solo la posterior puntuación crea los resultados finales. La latencia mide el consumo completo del generador de Whisper, excluye la carga del modelo y registra CPU/GPU. La comparación no aísla el efecto del hardware.

In [2]:
RECALCULAR_MODELOS = False
if RECALCULAR_MODELOS:
    generar_predicciones('audio')
    generar_predicciones('documento')
predicciones = leer_csv(PRED, PRED_COLS)
if not predicciones.empty:
    tiempos = predicciones.assign(latencia_seg=pd.to_numeric(predicciones.latencia_seg)).groupby(['modalidad','modelo','dispositivo']).agg(casos=('id','nunique'), latencia_total_seg=('latencia_seg','sum')).reset_index()
    display(tiempos)
else:
    print('Todavía no hay predicciones de comparación guardadas.')


,modalidad,modelo,dispositivo,casos,latencia_total_seg
0,audio,Qwen3-ASR-0.6B,CUDA RTX 4060 Laptop,10,131.907703
1,audio,faster-whisper-small-CPU,CPU,10,126.492207
2,documento,EasyOCR-1.7.2-es-en,CPU,10,394.289163
3,documento,PaddleOCR-3.7-PP-OCRv6,CPU,10,657.155718


## CER y WER sobre las mismas referencias humanas

Normalización idéntica: Unicode NFC, minúsculas, signos a espacios, espacios compactados, conservación de tildes y exclusión de etiquetas `[H1]`/`[HABLANTE_1]`. Se suman sustituciones, inserciones y omisiones sobre todos los casos y se dividen entre los caracteres/palabras totales de referencia. No se promedian tasas individuales. `RTF = latencia / duración`; se informa también el RTF global ponderado por duración.

Las distancias se calculan con los originales locales antes de seudonimizar. Después se protegen los textos de exportación. Los conteos `errores` y `unidades_referencia` permiten recomponer las tasas agregadas sin publicar nombres. El diccionario de privacidad no forma parte de la normalización de evaluación.

In [3]:
resumen = pd.DataFrame()
try:
    resumen = puntuar_predicciones()
    display(resumen)
except ValueError as pendiente:
    print('EVALUACIÓN FINAL PENDIENTE:', pendiente)


,modalidad,modelo,metrica,valor,n,latencia_total_seg,rtf_medio,rtf_global,dispositivo
0,audio,Qwen3-ASR-0.6B,WER,0.443038,10,131.907703,0.115765,0.117631,CUDA RTX 4060 Laptop
1,audio,faster-whisper-small-CPU,WER,0.409494,10,126.492207,0.108221,0.112802,CPU
2,documento,EasyOCR-1.7.2-es-en,CER,0.401108,10,394.289163,,,CPU
3,documento,PaddleOCR-3.7-PP-OCRv6,CER,0.211788,10,657.155718,,,CPU


In [4]:
if not resumen.empty:
    for metrica in ['CER','WER']:
        z=resumen[resumen.metrica==metrica]
        display(px.bar(z,x='modelo',y='valor',text='valor',title=f'{metrica}: comparación sobre las mismas 10 referencias'))
else:
    print('No hay CER/WER finales hasta completar las 20 referencias manuales.')


## Extracción estructurada y auditoría de 20 casos

Se exige modalidad, tema y resumen. Emisor/fecha/monto y citas/número de hablantes solo pueden aparecer si están sustentados; los ausentes deben ser `null`. La inferencia local puede preceder a la revisión, pero sus borradores no se publican hasta la validación humana. La auditoría usa una muestra fija de 10 documentos y 10 audios y cuenta como correcto solo el caso cuyos campos sean todos correctos.

In [5]:
RECALCULAR_LLM = False
if RECALCULAR_LLM: extraer_llm('ollama')
ext = leer_csv(EXTRACCIONES, ['id','modalidad','campos_json','modelo_llm','fecha'])
print('Extracciones LLM reales:', len(ext))
if not ext.empty:
    display(ext.groupby(['modalidad','modelo_llm']).size().rename('extracciones').reset_index())
    auditoria = preparar_auditoria()
    display(auditoria[['id','modalidad','completamente_correcto','observacion_auditor']])
try:
    n,total,acuerdo=acuerdo_llm()
    print(f'Acuerdo LLM-humano: {n}/{total} = {acuerdo:.1%}')
except (ValueError,FileNotFoundError) as pendiente:
    print('AUDITORÍA HUMANA PENDIENTE:', pendiente)


Extracciones LLM reales: 47


,modalidad,modelo_llm,extracciones
0,audio,qwen2.5:3b,22
1,documento,qwen2.5:3b,25


,id,modalidad,completamente_correcto,observacion_auditor
0,doc_001,documento,si,
1,doc_023,documento,si,
2,doc_017,documento,si,
3,doc_021,documento,si,
4,doc_015,documento,si,
5,doc_005,documento,si,
6,doc_012,documento,si,
7,doc_006,documento,si,
8,doc_010,documento,no,Decisión del estudiante confirmada en conversa...
9,doc_004,documento,si,


Acuerdo LLM-humano: 14/20 = 70.0%


## Casos difíciles y decisión

Las fotos incorporan reflejos, sombras y perspectiva; los manuscritos incluyen caligrafía histórica y tinta del reverso. Los audios naturales contienen ruido callejero. El análisis de precisión por tipo solo se completa con referencias manuales, conservando los mismos IDs para todos los modelos. La decisión debe considerar CER/WER, latencia, memoria y el coste de mantenimiento; no se exige que el alternativo gane.

In [6]:
r=leer_csv(MODELOS, PRED_COLS+['cer','wer'])
if not r.empty:
    d=leer_csv(DOC_CSV)
    por_tipo=r[r.modalidad=='documento'].merge(d[['id','tipo']],on='id',validate='many_to_one')
    display(por_tipo[['id','tipo','modelo','cer','latencia_seg']])
    grupos=por_tipo.groupby(['tipo','modelo'])[['errores','unidades_referencia']].agg(lambda x:pd.to_numeric(x).sum())
    grupos['CER']=grupos.errores/grupos.unidades_referencia
    display(grupos)
    display(r[r.modalidad=='audio'][['id','modelo','wer','latencia_seg','rtf','dispositivo']])
else:
    print('Reflexión de precisión y elección final pendientes de CER/WER reales; no se inventa una conclusión.')


,id,tipo,modelo,cer,latencia_seg
0,doc_005,impreso,EasyOCR-1.7.2-es-en,0.31400602409638556,16.236837899996317
1,doc_001,impreso,EasyOCR-1.7.2-es-en,0.4009077155824508,14.78585380000004
2,doc_004,impreso,EasyOCR-1.7.2-es-en,0.21875,15.189444199997524
3,doc_013,foto,EasyOCR-1.7.2-es-en,0.6848184818481848,64.87676730000385
4,doc_009,foto,EasyOCR-1.7.2-es-en,0.053613053613053616,36.67259529999865
5,doc_010,foto,EasyOCR-1.7.2-es-en,0.10213032581453634,53.61631900000066
6,doc_018,manuscrito,EasyOCR-1.7.2-es-en,0.6519114688128773,33.49171060000663
7,doc_021,manuscrito,EasyOCR-1.7.2-es-en,0.7504,51.73366060000262
8,doc_022,manuscrito,EasyOCR-1.7.2-es-en,0.8141361256544503,56.97145919999457
9,doc_023,manuscrito,EasyOCR-1.7.2-es-en,0.7005494505494505,50.71451540000271


errores  unidades_referencia       CER
tipo       modelo                                                        
foto       EasyOCR-1.7.2-es-en         624                 3060  0.203922
           PaddleOCR-3.7-PP-OCRv6      316                 3060  0.103268
impreso    EasyOCR-1.7.2-es-en        1039                 3621  0.286937
           PaddleOCR-3.7-PP-OCRv6      485                 3621  0.133941
manuscrito EasyOCR-1.7.2-es-en        2318                 3244  0.714550
           PaddleOCR-3.7-PP-OCRv6     1301                 3244  0.401048

,id,modelo,wer,latencia_seg,rtf,dispositivo
0,clip_021,Qwen3-ASR-0.6B,1.251700680272109,18.990534399999888,0.13932778911399216,CUDA RTX 4060 Laptop
1,clip_022,Qwen3-ASR-0.6B,0.8082706766917294,24.407669199994416,0.12913397547156313,CUDA RTX 4060 Laptop
2,clip_018,Qwen3-ASR-0.6B,0.6201550387596899,12.138720700000704,0.11958837363237555,CUDA RTX 4060 Laptop
3,clip_005,Qwen3-ASR-0.6B,0.6153846153846154,10.447916400000395,0.10653264736078773,CUDA RTX 4060 Laptop
4,clip_019,Qwen3-ASR-0.6B,0.1836734693877551,10.201761599993915,0.10300062824575962,CUDA RTX 4060 Laptop
5,clip_001,Qwen3-ASR-0.6B,0.3503184713375796,11.60935910000262,0.11491350116401004,CUDA RTX 4060 Laptop
6,clip_006,Qwen3-ASR-0.6B,0.2875816993464052,11.315476199997647,0.11342364693298698,CUDA RTX 4060 Laptop
7,clip_012,Qwen3-ASR-0.6B,0.011904761904761904,11.255968099998427,0.11399718550925599,CUDA RTX 4060 Laptop
8,clip_016,Qwen3-ASR-0.6B,0.05521472392638037,11.008666100002301,0.11207372820102722,CUDA RTX 4060 Laptop
9,clip_010,Qwen3-ASR-0.6B,0.136986301369863,10.53163090000453,0.10565698674668665,CUDA RTX 4060 Laptop


### Decisión con los resultados actuales

Se mantiene PaddleOCR y Whisper small como motores principales. En los originales y referencias disponibles, PaddleOCR tuvo CER 21,18 % frente a 40,11 % de EasyOCR, con mayor latencia total (657 frente a 394 s). El coste de esa precisión es más tiempo de CPU. Whisper small tuvo WER 40,95 % frente a 44,30 % de Qwen3-ASR y RTF global 0,113 frente a 0,118; la comparación mezcla CPU y GPU y no acredita superioridad universal.

El CER más alto se concentró en manuscritos. En `clip_021`, las inserciones elevaron WER por encima del 100 %: no se recorta esta tasa. El estudiante volvió a escuchar y confirmó ambas referencias sin cambios. Se conservan sus decisiones; las diferencias de abreviación también penalizan WER. No se ajustan parámetros después de observar el test. Memoria máxima por proceso no medida. La elección final debe revisarse si las correcciones cambian estos resultados.

La auditoría humana registrada da 14/20 casos completamente correctos (70 %). Los seis errores incluyen una fecha cambiada y números de hablantes sin marcas suficientes. Las decisiones confirmadas por conversación se registran con origen y fecha; las respuestas del modelo se mantienen sin corrección.

In [7]:
from validar_publicacion import verificar
try:
    verificar()
    print('Textos revisados y aptos para preparar publicación. Comprobar también ejemplos y evidencias.')
except ValueError as pendiente:
    print('PUBLICACIÓN PENDIENTE:', pendiente)


Textos revisados y aptos para preparar publicación. Comprobar también ejemplos y evidencias.
